In [ ]:
import sys
sys.path.insert(0, '../backend')
import pdfplumber
from src.chord_engine.parser import classify_line, pdf_to_bkcp, extract_metadata
from src.chord_engine.transposer import transpose_bkcp, get_key_name

PDF_PATH = '../bandkit-data/songs/ze_ramalho_sinonimos.pdf'  # ← edite aqui

In [ ]:
with pdfplumber.open(PDF_PATH) as pdf:
    for i, page in enumerate(pdf.pages):
        print(f'=== PÁGINA {i+1} ===')
        print(page.extract_text())
        print()
# SE o texto estiver ilegível ou vazio: o PDF pode ser escaneado.
# Traga o output para análise antes de continuar.

In [ ]:
with pdfplumber.open(PDF_PATH) as pdf:
    text = pdf.pages[0].extract_text()
for line in text.splitlines():
    tipo = classify_line(line)
    marker = '🎵' if tipo == 'chord' else '📝' if tipo == 'lyric' else '📌' if tipo == 'section' else '  '
    print(f'{marker} [{tipo:8}]  {line}')
# Verifique se linhas de acordes foram classificadas como 'chord'
# e linhas de letra como 'lyric'. Se algo estiver errado, reporte.

In [ ]:
bkcp_content, metadata = pdf_to_bkcp(PDF_PATH)
print('=== METADADOS ===')
for k, v in metadata.items():
    print(f'  {k}: {v}')
print()
print('=== CONTEÚDO .bkcp (primeiras 50 linhas) ===')
for line in bkcp_content.splitlines()[:50]:
    print(line)

In [ ]:
original_key = metadata.get('key', 'Am')
print(f'Tom original: {original_key}')
print()
for semitones in [1, 2, 3, -1, -2]:
    transposed = transpose_bkcp(bkcp_content, semitones)
    new_key = get_key_name(original_key, semitones)
    print(f'=== {semitones:+d} semitons → {new_key} ===')
    chord_lines = [l for l in transposed.splitlines() if '[' in l][:5]
    for l in chord_lines:
        print(f'  {l}')
    print()

In [ ]:
lines = bkcp_content.splitlines()
chord_lines = [l for l in lines if '[' in l and l.strip()]
lyric_lines = [l for l in lines if '[' not in l and l.strip() and not l.startswith('{')]
total = len(chord_lines) + len(lyric_lines)
print(f'Linhas com acordes:  {len(chord_lines)}')
print(f'Linhas só letra:     {len(lyric_lines)}')
print(f'Cobertura acordes:   {len(chord_lines)/max(total,1):.0%}')
print()
print('Se cobertura < 50%: o parser pode estar classificando acordes como letra.')
print('Traga o output da Célula 3 para diagnóstico.')